# Sample ML Notebook

A practical, end-to-end notebook for data setup, cleaning, analysis, visualization, and baseline modeling.

## 1. Set Up Python Environment

Import core libraries, configure plotting style, and set a reproducible random seed.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

OUTPUT_DIR = Path("../visualizations/notebook_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 2. Create a Sample Dataset

Generate a small classification dataset, define feature and target columns, and preview records.

In [ ]:
X, y = make_classification(
    n_samples=600,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    random_state=42
)

feature_cols = [f"feature_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_cols)
df["target"] = y

# Add a simple group column for group-level analysis later.
df["segment"] = np.where(df["feature_0"] > 0, "A", "B")

# Inject small issues for cleaning demo.
df.loc[df.sample(frac=0.03, random_state=42).index, "feature_2"] = np.nan
df = pd.concat([df, df.iloc[:5]], ignore_index=True)

target_col = "target"
print(f"Shape: {df.shape}")
df.head()

## 3. Inspect and Clean Data

Check dtypes, missing values, duplicates, and apply simple cleaning.

In [ ]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicate_count}")

clean_df = df.copy()
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
clean_df["feature_2"] = clean_df["feature_2"].fillna(clean_df["feature_2"].median())
clean_df["segment"] = clean_df["segment"].astype("category")

print(f"\nShape after cleaning: {clean_df.shape}")
clean_df.head()

## 4. Run Basic Exploratory Analysis

Compute summary statistics, target balance, correlations, and group-level aggregates.

In [ ]:
print("Summary statistics:")
display(clean_df.describe(include="all"))

print("Target distribution:")
print(clean_df[target_col].value_counts(normalize=True).rename("proportion"))

corr = clean_df[feature_cols + [target_col]].corr(numeric_only=True)
print("\nTop target correlations:")
print(corr[target_col].sort_values(ascending=False))

segment_agg = (
    clean_df.groupby("segment", observed=False)[feature_cols + [target_col]]
    .mean(numeric_only=True)
    .round(3)
)
print("\nSegment-level averages:")
display(segment_agg)

## 5. Visualize Key Patterns

Create histograms, box plots, scatter plots, and a correlation heatmap.

In [ ]:
# Histogram
ax = clean_df[feature_cols[0]].hist(bins=30)
ax.set_title(f"Histogram of {feature_cols[0]}")
ax.figure.savefig(OUTPUT_DIR / "hist_feature_0.png", dpi=150, bbox_inches="tight")
plt.show()

# Box plot by target
plt.figure()
sns.boxplot(data=clean_df, x=target_col, y=feature_cols[1])
plt.title(f"{feature_cols[1]} by target")
plt.savefig(OUTPUT_DIR / "boxplot_feature_1_by_target.png", dpi=150, bbox_inches="tight")
plt.show()

# Scatter plot
plt.figure()
sns.scatterplot(data=clean_df, x=feature_cols[2], y=feature_cols[3], hue=target_col, alpha=0.7)
plt.title(f"{feature_cols[2]} vs {feature_cols[3]}")
plt.savefig(OUTPUT_DIR / "scatter_features_2_3.png", dpi=150, bbox_inches="tight")
plt.show()

# Correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Train a Simple Baseline Model

Split data, train a baseline classifier, evaluate with standard metrics.

In [ ]:
X_data = clean_df[feature_cols]
y_data = clean_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X_data,
    y_data,
    test_size=0.2,
    random_state=42,
    stratify=y_data
)

baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=500, random_state=42))
])

baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

print("Classification report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

## 7. Export Results and Artifacts

Save cleaned data, predictions, and model outputs for reuse.

In [ ]:
import pickle

clean_path = OUTPUT_DIR / "cleaned_sample_data.csv"
pred_path = OUTPUT_DIR / "baseline_predictions.csv"
model_path = OUTPUT_DIR / "baseline_logreg_model.pkl"

clean_df.to_csv(clean_path, index=False)

prediction_df = X_test.copy()
prediction_df["actual"] = y_test.values
prediction_df["predicted"] = y_pred
prediction_df["predicted_probability"] = y_prob
prediction_df.to_csv(pred_path, index=False)

with open(model_path, "wb") as f:
    pickle.dump(baseline_model, f)

print("Saved artifacts:")
print(f"- {clean_path}")
print(f"- {pred_path}")
print(f"- {model_path}")